# Multi-Layer Perceptron (Neural Network)

An MLP is a **feedforward neural network** trained with backpropagation.  
Each layer applies a linear transformation followed by a non-linear activation.

**Dataset:** Scikit-learn Digits (8×8 = 64 features, 10 classes: digits 0–9)

**Architecture Used:**  
```
Input (64) → Hidden1 (128, sigmoid) → Hidden2 (64, sigmoid) → Output (10, softmax)
```

**Training:** Mini-batch gradient descent + cross-entropy loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, r"/home/claude/project/2026_Data_Science_and_Machine_Learning/src/rice_ml/supervised_learning")
from multi_layer_perceptron import multi_layer_perceptron
np.random.seed(42)
print("Imports complete")

## Load the Dataset

We use sklearn's built-in **Digits dataset** — handwritten digits (0–9) as 8×8 grayscale images, each flattened into a 64-dimensional feature vector. This is equivalent to a small MNIST.

**Why this dataset?**  
- 10 classes (non-linearly separable) — a good stress test for MLPs  
- Fast to train; no GPU required  
- Structured enough to get >95% accuracy with a small network

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target
print(f"Dataset shape: {X.shape}  |  Classes: {np.unique(y)}")
print(f"Pixel value range: [{X.min():.0f}, {X.max():.0f}]")

## Exploratory Data Analysis

Visualising sample digits confirms the dataset structure and shows the variety within each class.

In [ ]:
# Visualise sample images per class
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    examples = np.where(y == digit)[0]
    for row, idx in enumerate(examples[:2]):
        axes[row, digit].imshow(X[idx].reshape(8, 8), cmap='gray_r', interpolation='nearest')
        axes[row, digit].set_title(f"'{digit}'", fontsize=10) if row == 0 else None
        axes[row, digit].axis('off')
plt.suptitle("Sample Digit Images (2 per class)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Class distribution
fig, ax = plt.subplots(figsize=(10, 4))
counts = np.bincount(y)
bars = ax.bar(range(10), counts, color=plt.cm.tab10(np.linspace(0, 1, 10)), edgecolor='k', alpha=0.85)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', fontsize=10, fontweight='bold')
ax.set_xlabel("Digit Class", fontsize=12)
ax.set_ylabel("Sample Count", fontsize=12)
ax.set_title("Class Distribution", fontsize=13, fontweight='bold')
ax.set_xticks(range(10))
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Preprocessing & Split

We normalise pixel values to [0, 1] and use a stratified 80/20 train/test split.

In [ ]:
# Normalise to [0, 1]
X_norm = X / 16.0

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.20, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train class distribution: {np.bincount(y_train)}")

## Architecture Overview

We build a network: `64 → 128 → 64 → 10`  
- **Input:** 64 features (8×8 flattened pixels, normalised to [0,1])  
- **Hidden 1:** 128 neurons — sigmoid activation  
- **Hidden 2:** 64 neurons — sigmoid activation  
- **Output:** 10 neurons — softmax activation (class probabilities)

Weights are He-initialised (scale = √(2/fan_in)) for better gradient flow.

In [ ]:
model = multi_layer_perceptron(
    layer_sizes=[64, 128, 64, 10],
    learning_rate=0.05,
    epochs=30,
    batch_size=32,
    random_state=42
)

model.fit(X_train, y_train)
print(f"Training complete. Final loss: {model.loss_history[-1]:.4f}")

## Training Loss Curve

The loss curve documents how cross-entropy decreases epoch by epoch. Rapid early drops indicate effective gradient steps; flattening near zero is the target.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
axes[0].plot(range(1, len(model.loss_history) + 1), model.loss_history,
             color='steelblue', linewidth=2.5, marker='o', markersize=4)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Cross-Entropy Loss", fontsize=12)
axes[0].set_title("Training Loss Curve", fontsize=13, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Loss reduction per epoch
delta = np.diff(model.loss_history)
axes[1].bar(range(1, len(delta) + 1), -delta, color='#2a9d8f', edgecolor='k', alpha=0.75)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Loss Reduction per Epoch", fontsize=12)
axes[1].set_title("Learning Speed per Epoch", fontsize=13, fontweight='bold')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Evaluate

We measure accuracy on both train and test sets and display the full per-class classification report.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test, y_test)
y_pred    = model.predict(X_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test  Accuracy: {test_acc:.4f}")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

## Confusion Matrix

The confusion matrix reveals which digit pairs the network confuses most often. Off-diagonal entries highlight systematic misclassification patterns.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(10)); axes[0].set_yticks(range(10))
axes[0].set_xlabel("Predicted Digit", fontsize=12)
axes[0].set_ylabel("Actual Digit", fontsize=12)
axes[0].set_title("Confusion Matrix", fontsize=13, fontweight='bold')
for i in range(10):
    for j in range(10):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=8,
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=axes[0])

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
bars = axes[1].bar(range(10), per_class_acc, color=plt.cm.tab10(np.linspace(0, 1, 10)),
                   edgecolor='k', alpha=0.85)
for bar, acc in zip(bars, per_class_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{acc:.2f}", ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(range(10))
axes[1].set_xlabel("Digit Class", fontsize=12)
axes[1].set_ylabel("Per-Class Accuracy", fontsize=12)
axes[1].set_title("Accuracy per Digit", fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1.12)
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Visualise Predictions on Test Samples

Displaying sample test images with their true and predicted labels immediately reveals which examples the network gets right and wrong.

In [ ]:
# Show 20 test samples with true/predicted labels
probas = model.predict_proba(X_test)
confidence = np.max(probas, axis=1)
correct_mask = y_pred == y_test

# Pick 10 correct and 10 incorrect if available
correct_idx   = np.where(correct_mask)[0][:10]
incorrect_idx = np.where(~correct_mask)[0][:10]
show_idx = np.concatenate([correct_idx, incorrect_idx])

n_show = len(show_idx)
fig, axes = plt.subplots(2, 10, figsize=(18, 5))
for i, idx in enumerate(show_idx):
    row, col = divmod(i, 10)
    axes[row, col].imshow(X_test[idx].reshape(8, 8), cmap='gray_r', interpolation='nearest')
    color = '#2a9d8f' if correct_mask[idx] else '#e63946'
    axes[row, col].set_title(
        f"T:{y_test[idx]} P:{y_pred[idx]}\n{confidence[idx]:.2f}",
        fontsize=8, color=color, fontweight='bold')
    axes[row, col].axis('off')
axes[0, 0].set_ylabel("Correct", fontsize=11, labelpad=5)
axes[1, 0].set_ylabel("Wrong", fontsize=11, labelpad=5, color='#e63946')
plt.suptitle("Test Predictions (green=correct, red=wrong) | T=True, P=Predicted", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Effect of Architecture — Depth & Width

We compare four architectures to see how depth and width affect test accuracy.

In [ ]:
configs = {
    'Shallow [64,32,10]':      [64, 32, 10],
    'Default [64,128,64,10]':  [64, 128, 64, 10],
    'Deep [64,64,64,64,10]':   [64, 64, 64, 64, 10],
    'Wide [64,256,10]':        [64, 256, 10],
}

results = {}
loss_histories = {}
for name, sizes in configs.items():
    m = multi_layer_perceptron(layer_sizes=sizes, learning_rate=0.05,
                               epochs=25, batch_size=32, random_state=42)
    m.fit(X_train, y_train)
    results[name] = m.score(X_test, y_test)
    loss_histories[name] = m.loss_history

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curves
colors_arch = ['#e63946', '#457b9d', '#2a9d8f', '#f4a261']
for (name, hist), color in zip(loss_histories.items(), colors_arch):
    axes[0].plot(range(1, len(hist) + 1), hist, label=name, linewidth=2, color=color)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Cross-Entropy Loss", fontsize=12)
axes[0].set_title("Training Loss by Architecture", fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Accuracy bar chart
bars = axes[1].bar(range(len(results)), list(results.values()),
                   color=colors_arch, edgecolor='k', alpha=0.85)
for bar, (name, acc) in zip(bars, results.items()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f"{acc:.3f}", ha='center', fontsize=11, fontweight='bold')
axes[1].set_xticks(range(len(results)))
axes[1].set_xticklabels(list(results.keys()), rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel("Test Accuracy", fontsize=12)
axes[1].set_ylim(0.8, 1.05)
axes[1].set_title("Architecture Comparison — Test Accuracy", fontsize=13, fontweight='bold')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
print("Accuracy results:", {k: f"{v:.4f}" for k, v in results.items()})

## Key Takeaways

- MLPs learn hierarchical representations through stacked layers.
- **Sigmoid** (hidden layers) combined with **softmax** (output) handles multi-class classification.
- Mini-batch gradient descent balances computation speed and convergence stability.
- Deeper and wider networks can improve accuracy but may overfit on small datasets.
- **Backpropagation** efficiently computes gradients using the chain rule.